In [1]:
# To support both python 2 and python 3
from __future__ import division, print_function, unicode_literals

# Common imports
import numpy as np
import os

In [2]:
# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "classification"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [3]:
base_dir = './UAV_Classification_Dataset/dataset/MFCC'

# taking a subset of data to start

class_names = [
     'DJI_Phantom2',
     'UDI_U46',
     'Syma_X20',
     'Hasakee_Q11',
     'Hover_X1',
     'PhenoBee',
     'DJI_Matrice200_v2',
     'DJI_Mini3_pro',
     'Yuneec_Typhoon_H_Plus',
     'DJI_Phantom4',
     'DJI_Tello_TT',
     'DJI_FPV',
     'Syma_X26',
     'DJI_Tello',
     'DJI_Matrice600p',
     'DJI_Neo',
     'Holystone_HS210',
     'DJI_Avata',
     'DJI_Mavic2s',
     'DJI_Mavic_Air2',
     'Autel_Evo_II',
     'DJI_Mini3',
     'DJI_Mavic_Mini1',
     'DJI_Mavic_Mini2',
     'Syma_X8SW',
     'Syma_X5UW',
     'DJI_Mavic2pro',
     'DJI_Matrice200',
     'Syma_X20P',
     'David_Tricopter',
     'Swellpro_Splash3_plus',
     'Syma_X5SW'
]
os.listdir(base_dir)

['DJI_Phantom2',
 'UDI_U46',
 'Syma_X20',
 'Hasakee_Q11',
 'Hover_X1',
 'PhenoBee',
 'DJI_Matrice200_v2',
 'DJI_Mini3_pro',
 'Yuneec_Typhoon_H_Plus',
 'DJI_Phantom4',
 'DJI_Tello_TT',
 'DJI_FPV',
 'Syma_X26',
 'DJI_Tello',
 'DJI_Matrice600p',
 'DJI_Neo',
 'Holystone_HS210',
 'DJI_Avata',
 'DJI_Mavic2s',
 'DJI_Mavic_Air2',
 'Autel_Evo_II',
 'DJI_Mini3',
 'DJI_Mavic_Mini1',
 'DJI_Mavic_Mini2',
 'Syma_X8SW',
 'Syma_X5UW',
 'DJI_Mavic2pro',
 'DJI_Matrice200',
 'Syma_X20P',
 'David_Tricopter',
 'Swellpro_Splash3_plus',
 'Syma_X5SW']

In [4]:
# ------------------------------------
# PREP DATA
# ------------------------------------

In [4]:
base_name = 'cases_resized'
width = 120

In [6]:
# getting number of instances per class
for class_name in class_names:
    class_folder = os.path.join(base_dir, class_name);
    files = [f for f in os.listdir(class_folder)]
    print(class_name + ": " + str(len(files)) + " instances in class")

DJI_Phantom2: 100 instances in class
UDI_U46: 100 instances in class
Syma_X20: 100 instances in class
Hasakee_Q11: 100 instances in class
Hover_X1: 100 instances in class
PhenoBee: 100 instances in class
DJI_Matrice200_v2: 100 instances in class
DJI_Mini3_pro: 100 instances in class
Yuneec_Typhoon_H_Plus: 100 instances in class
DJI_Phantom4: 100 instances in class
DJI_Tello_TT: 100 instances in class
DJI_FPV: 100 instances in class
Syma_X26: 100 instances in class
DJI_Tello: 100 instances in class
DJI_Matrice600p: 100 instances in class
DJI_Neo: 100 instances in class
Holystone_HS210: 100 instances in class
DJI_Avata: 100 instances in class
DJI_Mavic2s: 100 instances in class
DJI_Mavic_Air2: 100 instances in class
Autel_Evo_II: 100 instances in class
DJI_Mini3: 100 instances in class
DJI_Mavic_Mini1: 100 instances in class
DJI_Mavic_Mini2: 100 instances in class
Syma_X8SW: 100 instances in class
Syma_X5UW: 100 instances in class
DJI_Mavic2pro: 100 instances in class
DJI_Matrice200: 100

In [ ]:
# RESIZE IMAGES
import joblib
from skimage.io import imread
from skimage.transform import resize
import cv2
import numpy as np
import os

def resize_all(src, pklname, include, width=150, height=None):
    """
    load images from path, resize them and write them as arrays to a dictionary,
    together with labels and metadata. The dictionary is written to a pickle file
    named '{pklname}_{width}x{height}px.pkl'.

    Parameter
    ---------
    src: str
        path to data
    pklname: str
        path to output file
    width: int
        target width of the image in pixels
    include: set[str]
        set containing str
    """
    height = height if height is not None else width

    data = dict()
    data['description'] = 'resized ({0}x{1})MFCC extractions in rgb'.format(int(width), int(height))
    data['label'] = []
    data['filename'] = []
    data['data'] = []

    pklname = f"{pklname}_{width}x{height}px.pkl"

    # read all images in PATH, resize and write to DESTINATION_PATH
    for subdir in os.listdir(src):
        if subdir in include:
            print(subdir)
            current_path = os.path.join(src, subdir)

            for file in os.listdir(current_path):
                if file[-3:] in {'jpg', 'png'}:
                    im = cv2.imread(os.path.join(current_path, file))
                    im = cv2.resize(im, (width, height), interpolation=cv2.INTER_AREA) #[:,:,::-1]

                    # !!!!!!!!! extract base name to avoid labeling error !!!!!!!!!!
                    filename = os.path.basename(subdir)
                    data['label'].append(filename)
                    # data['label'].append(subdir[:-4])
                    data['filename'].append(file)
                    data['data'].append(im)

        joblib.dump(data, pklname)

In [6]:
# RESIZE IMAGES
!pip install imageio[pyav]

data_path = './UAV_Classification_Dataset/dataset/MFCC'
base_name = 'cases_resized'
width = 120
# ##### VERY SMALL VERSION ######
# include = {
#      'DJI_Phantom2',
#      'UDI_U46',
#      'Syma_X20',
#      'Hasakee_Q11',
#      'Hover_X1',
#      'PhenoBee',
#      'DJI_Matrice200_v2',
#      'DJI_Mini3_pro',
#      'Yuneec_Typhoon_H_Plus',
#      'DJI_Phantom4',
#      'DJI_Tello_TT',
#      'DJI_FPV',
#      'Syma_X26',
#      'DJI_Tello',
#      'DJI_Matrice600p',
#      'DJI_Neo',
#      'Holystone_HS210',
#      'DJI_Avata',
#      'DJI_Mavic2s',
#      'DJI_Mavic_Air2',
#      'Autel_Ev6o_II',
#      'DJI_Mini3',
#      'DJI_Mavic_Mini1',
#      'DJI_Mavic_Mini2',
#      'Syma_X8SW',
#      'Syma_X5UW',
#      'DJI_Mavic2pro',
#      'DJI_Matrice200',
#      'Syma_X20P',
#      'David_Tricopter',
#      'Swellpro_Splash3_plus',
#      'Syma_X5SW'
#  }
# resize_all(src=data_path, pklname=base_name, width=width, include=include)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
# load the data from disk and print a summary
import joblib
from collections import Counter
import numpy as np

data = joblib.load(f'{base_name}_{width}x{width}px.pkl')
 
print('number of samples: ', len(data['data']))
print('keys: ', list(data.keys()))
print('description: ', data['description'])
print('image shape: ', data['data'][0].shape)
print('labels:', np.unique(data['label']))
 
Counter(data['label'])

number of samples:  3100
keys:  ['description', 'label', 'filename', 'data']
description:  resized (120x120)MFCC extractions in rgb
image shape:  (120, 120, 3)
labels: ['DJI_Avata' 'DJI_FPV' 'DJI_Matrice200' 'DJI_Matrice200_v2'
 'DJI_Matrice600p' 'DJI_Mavic2pro' 'DJI_Mavic2s' 'DJI_Mavic_Air2'
 'DJI_Mavic_Mini1' 'DJI_Mavic_Mini2' 'DJI_Mini3' 'DJI_Mini3_pro' 'DJI_Neo'
 'DJI_Phantom2' 'DJI_Phantom4' 'DJI_Tello' 'DJI_Tello_TT'
 'David_Tricopter' 'Hasakee_Q11' 'Holystone_HS210' 'Hover_X1' 'PhenoBee'
 'Swellpro_Splash3_plus' 'Syma_X20' 'Syma_X20P' 'Syma_X26' 'Syma_X5SW'
 'Syma_X5UW' 'Syma_X8SW' 'UDI_U46' 'Yuneec_Typhoon_H_Plus']


Counter({'DJI_Phantom2': 100,
         'UDI_U46': 100,
         'Syma_X20': 100,
         'Hasakee_Q11': 100,
         'Hover_X1': 100,
         'PhenoBee': 100,
         'DJI_Matrice200_v2': 100,
         'DJI_Mini3_pro': 100,
         'Yuneec_Typhoon_H_Plus': 100,
         'DJI_Phantom4': 100,
         'DJI_Tello_TT': 100,
         'DJI_FPV': 100,
         'Syma_X26': 100,
         'DJI_Tello': 100,
         'DJI_Matrice600p': 100,
         'DJI_Neo': 100,
         'Holystone_HS210': 100,
         'DJI_Avata': 100,
         'DJI_Mavic2s': 100,
         'DJI_Mavic_Air2': 100,
         'DJI_Mini3': 100,
         'DJI_Mavic_Mini1': 100,
         'DJI_Mavic_Mini2': 100,
         'Syma_X8SW': 100,
         'Syma_X5UW': 100,
         'DJI_Mavic2pro': 100,
         'DJI_Matrice200': 100,
         'Syma_X20P': 100,
         'David_Tricopter': 100,
         'Swellpro_Splash3_plus': 100,
         'Syma_X5SW': 100})

In [6]:
# name input data X and result (labels) y
X = np.array(data['data'])
y = np.array(data['label'])

In [ ]:
# save the dictionary of resize images to a npy file
# np.save("drone_mfccs_120.npy", data)

# TO LOAD THIS:
# read_dict = np.load("drone_mfccs_128.npy", allow_pickle='TRUE').item()
# print(read_dict['DJI_Avata'] # displays this class data

In [ ]:
# ------------------------------------------------
# DEVELOP CNN NETWORK 
# https://www.tensorflow.org/tutorials/images/cnn
# ------------------------------------------------

In [8]:
import tensorflow as tf

from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

In [ ]:
# print(X_train.shape, X_test.shape)

In [10]:
# ---------------------------------- set up strat k fold -----------------------------------------------------------------

from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

# NORMALIZE X PIXEL VALS TO BE BTWN 0 AND 1
X = X / 255.0 

# Lets split the data into 5 folds.  
# We will use this 'kf'(KFold splitting stratergy) object as input to cross_val_score() method
kf =KFold(n_splits=5, shuffle=True, random_state=42)

for i, (train, val) in enumerate(kf.split(X)):
    print('split {}:'.format(i))
    print('training indices: {}'.format(train))
    print('validation indices: {}'.format(val))
    print()

split 0:
training indices: [   1    2    3 ... 3096 3097 3098]
validation indices: [   0   14   22   25   29   30   31   32   43   44   45   48   51   52
   56   63   67   70   73   80   87   88   93  102  108  111  120  124
  134  135  139  141  144  149  152  162  170  173  174  175  176  177
  178  183  184  188  192  194  196  203  211  214  218  221  233  239
  240  246  251  254  256  257  266  270  272  282  289  291  296  298
  299  309  313  314  318  321  322  324  325  331  332  343  346  350
  354  361  366  368  370  387  393  402  408  410  411  414  416  420
  423  430  432  433  436  438  439  443  449  450  457  460  463  471
  472  478  479  485  486  495  506  507  511  521  527  528  532  533
  535  543  544  554  555  557  564  565  567  568  572  581  582  594
  598  599  601  602  605  610  611  612  621  637  642  651  670  679
  680  682  685  691  693  695  700  705  718  727  729  741  755  756
  759  761  764  765  772  776  785  790  794  798  801  805  807

In [11]:
import pandas as pd
import tensorflow as tf
import keras
from keras import layers, models, callbacks
from sklearn.preprocessing import LabelEncoder
import gc
import math

K=5
kf = KFold(n_splits=K, random_state=42, shuffle=True)
train_error_results = np.zeros(K)
val_error_results = np.zeros(K)
tf.random.set_seed(42)

mean_score = 0 # auto mean score calculation

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# def learning rate schedule
def lr_schedule(epoch):
    if epoch < 10:
        return 0.001
    else:
        return 0.001 * math.exp(0.1 * (10 - epoch))

for i, (train_ind, val_ind) in enumerate(kf.split(X)):
    print(f'--- Starting Split {i} ---')
    
    X_train_i, X_val = X[train_ind], X[val_ind]
    y_train_i, y_val = y_encoded[train_ind], y_encoded[val_ind]
    
    model = keras.Sequential()
    model.add(keras.Input(shape=(120, 120, 3)))
    
    model.add(layers.Conv2D(16, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2))) 
    
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Flatten())
    
    model.add(layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(32))
    model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

    # fit the LR model.
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    lr_callback = callbacks.LearningRateScheduler(lr_schedule)
    
    history = model.fit(X_train_i, y_train_i, epochs=30, batch_size=8,
                        callbacks=[lr_callback, early_stop],
                        validation_data=(X_val, y_val), verbose=1)
    
    print('Split {}'.format(i))
    
    print("\nTRAINING SET")
    print("Score:")
    train_loss, train_acc = model.evaluate(X_train_i, y_train_i)
    print(f"Loss: {train_loss}, Accuracy: {train_acc}\n")
    print()
    
    print("\nTEST SET")
    print("Score:")
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Loss: {val_loss}, Accuracy: {val_acc}\n")
    print() 

    mean_score += val_acc # for mean score calculation auto

    # memory cleanup
    del model
    del history
    tf.keras.backend.clear_session()
    gc.collect()

print("Mean Validation Accuracy:")
print(mean_score / K) # mean score validation set

--- Starting Split 0 ---


I0000 00:00:1778525443.308847   35978 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3623 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoch 1/30


2026-05-11 14:50:46.104797: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 428544000 exceeds 10% of free system memory.
2026-05-11 14:50:46.844009: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 428544000 exceeds 10% of free system memory.
I0000 00:00:1778525450.194466   36300 service.cc:152] XLA service 0x7f4004002f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778525450.194847   36300 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6
2026-05-11 14:50:50.358108: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778525450.961073   36300 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/310 ━━━━━━━━━━━━━━━━━━━━ 1:25:47 17s/step - accuracy: 0.1250 - loss: 8.1694

...

310/310 ━━━━━━━━━━━━━━━━━━━━ 33s 52ms/step - accuracy: 0.1195 - loss: 4.6491 - val_accuracy: 0.0290 - val_loss: 16.0229 - learning_rate: 0.0010
Epoch 2/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.3568 - loss: 2.4275 - val_accuracy: 0.4484 - val_loss: 2.7368 - learning_rate: 0.0010
Epoch 3/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.6531 - loss: 1.4914 - val_accuracy: 0.8339 - val_loss: 0.9332 - learning_rate: 0.0010
Epoch 4/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.7917 - loss: 1.0660 - val_accuracy: 0.9387 - val_loss: 0.5998 - learning_rate: 0.0010
Epoch 5/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.8693 - loss: 0.8351 - val_accuracy: 0.9113 - val_loss: 0.7643 - learning_rate: 0.0010
Epoch 6/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.8741 - loss: 0.9255 - val_accuracy: 0.9194 - val_loss: 0.7762 - learning_rate: 0.0010
Epoch 7/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.9127 - loss: 0.7867 - v

2026-05-11 14:52:35.469773: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 428544000 exceeds 10% of free system memory.
2026-05-11 14:52:36.946913: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 428544000 exceeds 10% of free system memory.


78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 62ms/step - accuracy: 0.9491 - loss: 0.5608
Loss: 0.5917128324508667, Accuracy: 0.9423387050628662



TEST SET
Score:
Loss: 0.5997600555419922, Accuracy: 0.9387096762657166


--- Starting Split 1 ---
Epoch 1/30


2026-05-11 14:52:53.165533: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 428544000 exceeds 10% of free system memory.


310/310 ━━━━━━━━━━━━━━━━━━━━ 21s 41ms/step - accuracy: 0.0684 - loss: 4.7487 - val_accuracy: 0.0323 - val_loss: 49.5907 - learning_rate: 0.0010
Epoch 2/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.0791 - loss: 3.5863 - val_accuracy: 0.1242 - val_loss: 3.3770 - learning_rate: 0.0010
Epoch 3/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.0998 - loss: 3.4024 - val_accuracy: 0.0371 - val_loss: 87.3949 - learning_rate: 0.0010
Epoch 4/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.1746 - loss: 3.1250 - val_accuracy: 0.3387 - val_loss: 2.4324 - learning_rate: 0.0010
Epoch 5/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.2570 - loss: 2.6379 - val_accuracy: 0.4823 - val_loss: 1.8367 - learning_rate: 0.0010
Epoch 6/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.4651 - loss: 1.9940 - val_accuracy: 0.7726 - val_loss: 0.9334 - learning_rate: 0.0010
Epoch 7/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.5841 - loss: 1.5629 

In [ ]:
# ----------------------------- 50 EPOCHS VERSION TO ALLOW FOR EARLY STOPPING TO WORK + NOT HIT CEILING ----------------------
import pandas as pd
import tensorflow as tf
import keras
from keras import layers, models, callbacks
from sklearn.preprocessing import LabelEncoder
import gc
import math

K=5
kf = KFold(n_splits=K, random_state=42, shuffle=True)
train_error_results = np.zeros(K)
val_error_results = np.zeros(K)
tf.random.set_seed(42)

mean_score = 0 # auto mean score calculation

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# def learning rate schedule
def lr_schedule(epoch):
    if epoch < 10:
        return 0.001
    else:
        return 0.001 * math.exp(0.1 * (10 - epoch))

for i, (train_ind, val_ind) in enumerate(kf.split(X)):
    print(f'--- Starting Split {i} ---')
    
    X_train_i, X_val = X[train_ind], X[val_ind]
    y_train_i, y_val = y_encoded[train_ind], y_encoded[val_ind]
    
    model = keras.Sequential()
    model.add(keras.Input(shape=(120, 120, 3)))
    
    model.add(layers.Conv2D(16, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2))) 
    
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Flatten())
    
    model.add(layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(32))
    model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

    # fit the LR model.
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    lr_callback = callbacks.LearningRateScheduler(lr_schedule)
    
    history = model.fit(X_train_i, y_train_i, epochs=50, batch_size=8,
                        callbacks=[lr_callback, early_stop],
                        validation_data=(X_val, y_val), verbose=1)
    
    print('Split {}'.format(i))
    
    print("\nTRAINING SET")
    print("Score:")
    train_loss, train_acc = model.evaluate(X_train_i, y_train_i)
    print(f"Loss: {train_loss}, Accuracy: {train_acc}\n")
    print()
    
    print("\nTEST SET")
    print("Score:")
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Loss: {val_loss}, Accuracy: {val_acc}\n")
    print() 

    mean_score += val_acc # for mean score calculation auto

    # memory cleanup
    del model
    del history
    tf.keras.backend.clear_session()
    gc.collect()

print("Mean Validation Accuracy:")
print(mean_score / K) # mean score validation set

--- Starting Split 0 ---
Epoch 1/50
310/310 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.1194 - loss: 4.9402 - val_accuracy: 0.0290 - val_loss: 29.7419 - learning_rate: 0.0010
Epoch 2/50
310/310 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.3782 - loss: 2.4586 - val_accuracy: 0.8016 - val_loss: 0.9456 - learning_rate: 0.0010
Epoch 3/50
310/310 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.6603 - loss: 1.5220 - val_accuracy: 0.7484 - val_loss: 1.2826 - learning_rate: 0.0010
Epoch 4/50
310/310 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.7637 - loss: 1.2032 - val_accuracy: 0.6516 - val_loss: 2.5022 - learning_rate: 0.0010
Epoch 5/50
310/310 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.8527 - loss: 0.9853 - val_accuracy: 0.6710 - val_loss: 2.7520 - learning_rate: 0.0010
Epoch 6/50
310/310 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.8952 - loss: 0.8462 - val_accuracy: 0.9113 - val_loss: 0.9172 - learning_rate: 0.0010
Epoch 7/50
310/310 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - 

In [11]:
# ------------------------------------ MOBILENETV2 FOR COMPARISON ------------------------------------------------------
# done w/ keras 
# -----------------------------------------------------------------------------------------------------------------

import numpy as np
import tensorflow as tf
from keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image

import pandas as pd
import keras
from keras import layers, models, callbacks
from sklearn.preprocessing import LabelEncoder
import gc
import math

K=5
kf = KFold(n_splits=K, random_state=42, shuffle=True)
train_error_results = np.zeros(K)
val_error_results = np.zeros(K)
tf.random.set_seed(42)

mean_score = 0 # auto mean score calculation

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# def learning rate schedule
def lr_schedule(epoch):
    if epoch < 10:
        return 0.001
    else:
        return 0.001 * math.exp(0.1 * (10 - epoch))

for i, (train_ind, val_ind) in enumerate(kf.split(X)):
    print(f'--- Starting Split {i} ---')
    
    X_train_i, X_val = X[train_ind], X[val_ind]
    y_train_i, y_val = y_encoded[train_ind], y_encoded[val_ind]
    
    base_model = MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_shape=(120, 120, 3)
    )
    base_model.trainable = False
    
    model = keras.Sequential()
    model.add(base_model)
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(32))
    
    model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
    # fit the LR model.
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    lr_callback = callbacks.LearningRateScheduler(lr_schedule)
    
    history = model.fit(X_train_i, y_train_i, epochs=30, batch_size=8,
                        callbacks=[lr_callback, early_stop],
                        validation_data=(X_val, y_val), verbose=1)
    
    print('Split {}'.format(i))
    
    print("\nTRAINING SET")
    print("Score:")
    train_loss, train_acc = model.evaluate(X_train_i, y_train_i)
    print(f"Loss: {train_loss}, Accuracy: {train_acc}\n")
    print()
    
    print("\nTEST SET")
    print("Score:")
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Loss: {val_loss}, Accuracy: {val_acc}\n")
    print() 

    mean_score += val_acc # for mean score calculation auto

    # memory cleanup
    del model
    del history
    tf.keras.backend.clear_session()
    gc.collect()

print("Mean Validation Accuracy:")
print(mean_score / K) # mean score validation set

--- Starting Split 0 ---


/tmp/ipykernel_2113/3707257162.py:41: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(
I0000 00:00:1778691416.339740    2113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3623 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


2026-05-13 12:56:59.967687: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 428544000 exceeds 10% of free system memory.
2026-05-13 12:57:00.847081: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 428544000 exceeds 10% of free system memory.


Epoch 1/30


...

78/78 ━━━━━━━━━━━━━━━━━━━━ 31s 206ms/step - accuracy: 0.9278 - loss: 0.3196
Loss: 0.35236451029777527, Accuracy: 0.914919376373291



TEST SET
Score:


2026-05-13 13:02:05.731041: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1167', 40 bytes spill stores, 40 bytes spill loads

2026-05-13 13:02:05.949084: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1167', 584 bytes spill stores, 584 bytes spill loads



Loss: 0.48783639073371887, Accuracy: 0.853225827217102


--- Starting Split 1 ---


2026-05-13 13:02:20.447464: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 428544000 exceeds 10% of free system memory.


Epoch 1/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 17s 40ms/step - accuracy: 0.1218 - loss: 3.7854 - val_accuracy: 0.4290 - val_loss: 1.7906 - learning_rate: 0.0010
Epoch 2/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.3890 - loss: 1.9962 - val_accuracy: 0.6323 - val_loss: 1.2900 - learning_rate: 0.0010
Epoch 3/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.5176 - loss: 1.5586 - val_accuracy: 0.6661 - val_loss: 1.1547 - learning_rate: 0.0010
Epoch 4/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.5831 - loss: 1.3271 - val_accuracy: 0.6790 - val_loss: 1.0226 - learning_rate: 0.0010
Epoch 5/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.6358 - loss: 1.1597 - val_accuracy: 0.7081 - val_loss: 0.9652 - learning_rate: 0.0010
Epoch 6/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.6430 - loss: 1.0476 - val_accuracy: 0.7032 - val_loss: 0.9426 - learning_rate: 0.0010
Epoch 7/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.6759 - loss: 1

In [ ]:
# -------------------------------- MODEL VERSION 3 --------------------------------------------
# Resource: https://pyimagesearch.com/2018/12/31/keras-conv2d-and-convolutional-layers/
# ---------------------------------------------------------------------------------------------
# Create convolutional base
import keras
from keras import layers

model = keras.Sequential()
model.add(keras.Input(shape=(120, 120, 3)))

model.add(layers.Conv2D(16, (3, 3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Flatten())

model.add(layers.Dropout(0.5))
model.add(layers.Dense(256, activation='relu'))

model.add(layers.Dense(32))

model.summary()

# consider adding padding = 'same'

In [ ]:
import tensorflow as tf
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=10, 
                    validation_data=(X_test, y_test))

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.0, 1])
plt.legend(loc='lower right')

test_loss, test_acc = model.evaluate(X_test,  y_test, verbose=2)

In [ ]:
# Assuming 'model' is your compiled Keras model and 'x_test', 'y_test' are your test data

# Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")